In [ ]:
from pathlib import Path
import json
import random

In [ ]:
def get_project_root() -> Path:
    current = Path.cwd()
    for candidate in [current, *current.parents]:
        if (candidate / "data_preprocess").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Unable to locate project root.")

PROJECT_ROOT = get_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "data_mixing"

base_token = 660_000
experiment_name = "exp1"

TOKEN_LIMITS = {
    "onehalf": int(base_token * 1.5),
}

DOMAIN_DATASETS = {
    "math": DATA_DIR / "openmathinstruct2_1M_len.json",
    "code": DATA_DIR / "opencoder-sft_len.json",
    "instr": DATA_DIR / "Infinity-Instruct_0625_len.json",
}

def sample_tokens(data, token_limit):
    selected = []
    total_tokens = 0
    for item in data:
        if total_tokens + item["len"] <= token_limit:
            total_tokens += item["len"]
            selected.append({k: v for k, v in item.items() if k != "len"})
        else:
            truncated_item = {k: v for k, v in item.items() if k != "len"}
            selected.append(truncated_item)
            break
    return selected

Sampled 2262 items for onehalf with 990000.0 tokens.


In [ ]:
experiment_dir = OUTPUT_DIR / experiment_name
experiment_dir.mkdir(parents=True, exist_ok=True)

for domain, dataset_path in DOMAIN_DATASETS.items():
    with dataset_path.open("r") as f:
        structured_data = json.load(f)

    random.shuffle(structured_data)

    for label, limit in TOKEN_LIMITS.items():
        sampled_items = sample_tokens(structured_data, limit)
        subset_path = experiment_dir / f"{base_token}_{domain}_{label}.json"
        with subset_path.open("w") as f:
            json.dump(sampled_items, f, indent=2)

        print(f"Saved {subset_path.name} with {len(sampled_items)} items")

Sampled 2168 items for onehalf with 990000.0 tokens.


Sampled 1873 items for onehalf with 990000.0 tokens.


In [ ]:
dataset_info_path = DATA_DIR / "dataset_info.json"

if dataset_info_path.exists():
    with dataset_info_path.open("r") as f:
        try:
            dataset_info = json.load(f)
        except json.JSONDecodeError:
            dataset_info = {}
else:
    dataset_info = {}

for domain in DOMAIN_DATASETS:
    for label in TOKEN_LIMITS:
        dataset_name = f"{base_token}_{domain}_{label}"
        relative_output = Path("data_mixing") / experiment_name / f"{base_token}_{domain}_{label}.json"
        dataset_info[dataset_name] = {"file_name": str(relative_output)}

with dataset_info_path.open("w") as f:
    json.dump(dataset_info, f, indent=2)

3000
